# Emission Spectroscopy with Equilibirum Chemistry

Last update: March 23th (2026) Hajime Kawahara for v2.1

In this getting started guide, we will use ExoJAX to simulate a high-resolution emission spectrum from an atmosphere with CO molecular absorption and hydrogen molecule CIA continuum absorption as the opacity sources. We assume the thermochemical equilibrium. We will then add appropriate noise to the simulated spectrum to create a mock spectrum and perform spectral retrieval using NumPyro's HMC NUTS. 

To compute Thermo Chemical Equilibrium (TCE), we use a differentiable TCE calculator, [ExoGibbs](https://github.com/HajimeKawahara/exogibbs) (v0.3.9) 

First, we recommend 64-bit if you do not think about numerical errors. Use jax.config to set 64-bit. 
(But note that 32-bit is sufficient in most cases. Consider to use 32-bit (faster, less device memory) for your real use case.) 

In [ ]:
from jax import config
config.update("jax_enable_x64", True)

## 1. Loading a molecular database using mdb

ExoJAX has an API for molecular databases, called `mdb` (or `adb` for atomic datbases). Prior to loading the database, define the wavenumber range first.

In [ ]:
from exojax.utils.grids import wavenumber_grid

nu_grid, wav, resolution = wavenumber_grid(
    22920.0, 23000.0, 3500, unit="AA", xsmode="premodit"
)
print("Resolution=", resolution)

Then, let's load the molecular database. We here use Carbon monoxide in Exomol. `CO/12C-16O/Li2015` means `Carbon monoxide/ isotopes = 12C + 16O / database name`. You can check the database name in the ExoMol website (https://www.exomol.com/).  

In [ ]:
from exojax.database.exomol.api import MdbExomol
mdb = MdbExomol(".database/CO/12C-16O/Li2015", nurange=nu_grid)

## 2. Computation of the Cross Section using opa

ExoJAX has various opacity calculator classes, so-called `opa`. Here, we use a memory-saved opa, `OpaPremodit`. We assume the robust tempreature range we will use is 500-1500K.

In [ ]:
from exojax.opacity import OpaPremodit
# trange must cover temperatures across the full ML pressure grid (1e-7 to 100 bar).
# e.g. powerlaw T0=1200, alpha=0.1 gives ~240 K at 1e-7 bar and ~1900 K at 100 bar.
opa = OpaPremodit(mdb, nu_grid, auto_trange=[200.0, 2500.0], dit_grid_resolution=1.0)


Then let's compute cross section for two different temperature 500 and 1500 K for P=1.0 bar. opa.xsvector can do that!

In [ ]:
P = 1.0  # bar
T_1 = 500.0  # K
xsv_1 = opa.xsvector(T_1, P)  # cm2

T_2 = 1500.0  # K
xsv_2 = opa.xsvector(T_2, P)  # cm2

Plot them. It can be seen that different lines are stronger at different temperatures.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(nu_grid, xsv_1, label=str(T_1) + "K")  # cm2
plt.plot(nu_grid, xsv_2, alpha=0.5, label=str(T_2) + "K")  # cm2
plt.yscale("log")
plt.legend()
plt.xlabel("wavenumber (cm-1)")
plt.ylabel("cross section (cm2)")
plt.show()

## 3. Atmospheric Radiative Transfer

ExoJAX can solve the radiative transfer and derive the emission spectrum. To do so, ExoJAX has `art` class. `ArtEmisPure` means Atomospheric Radiative Transfer for Emission with Pure absorption. So, `ArtEmisPure` does not include scattering.
We set the number of the atmospheric layer to 200 (nlayer) and the pressure at bottom and top atmosphere to 100 and 1.e-5 bar.

Since v1.5, one can choose the rtsolver (radiative transfer solver) from the flux-based 2 stream solver (`fbase2st`) and the intensity-based n-stream sovler (`ibased`). Use `rtsolver` option. In the latter case, the number of the stream (`nstream`) can be specified. Note that the default rtsolver for the pure absorption (i.e. no scattering nor reflection) has been `ibased` since v1.5.
In our experience, `ibased` is faster and more accurate than `fbased`.

In [ ]:
from exojax.rt import ArtEmisPure

# Pressure domain and nlayer must match the ML model training grid:
#   pressure_bottom_bar=100, pressure_top_bar=1e-7, num_levels=50
# Using a different nlayer or pressure range breaks the Transformer
# (wrong-length sequences produce VMR > 1 and physically impossible outputs).
art = ArtEmisPure(
    nu_grid=nu_grid,
    pressure_btm=1.0e2,
    pressure_top=1.0e-7,
    nlayer=50,
    rtsolver="ibased",
    nstream=8,
)


Let's assume the power law temperature model, within 500 - 1500 K.

$T = T_0 P^\alpha$

where $T_0=1200$ K and $\alpha=0.1$.

In [ ]:
art.change_temperature_range(200.0, 2500.0)
Tarr = art.powerlaw_temperature(1200.0, 0.1)


Sets Vulcan emulator

In [ ]:
from pathlib import Path
import sys, types
import numpy as np

BUNDLE_PATH = Path("best_exported.npz")

_src = bytes(np.load(BUNDLE_PATH, allow_pickle=False)["meta/vulcan_emulator_src"]).decode()
_mod = types.ModuleType("_embedded_vulcan_inference")
sys.modules["_embedded_vulcan_inference"] = _mod
exec(compile(_src, "<embedded>", "exec"), _mod.__dict__)
load_model, make_fastchem_vmr_fn = _mod.load_model, _mod.make_fastchem_vmr_fn

bundle = load_model(BUNDLE_PATH)
vmr_fn, species_labels = make_fastchem_vmr_fn(bundle)
print(f"Loaded: {bundle.chemistry_type} {bundle.model_type}  |  vmr_fn supports jit/grad/vjp/vmap")


Sets reference abundances for the ExoGibbs vs FastChem comparison

In [ ]:
from pathlib import Path
import jax.numpy as jnp

FASTCHEM_SOLAR_PATH = Path("/Users/imalsky/Desktop/Emulators/VULCAN_Project/VULCAN-master/fastchem_vulcan/input/solar_element_abundances.dat")

# Reference elemental number ratios relative to hydrogen (He/H, C/H, O/H,
# N/H, S/H) used as the 'solar' baseline for the ExoGibbs <-> FastChem-emulator
# comparison. These five elements are the exact global-input contract the
# FastChem bundle was trained against.
ML_SOLAR_ABUNDANCES = {
    "He_H": 8.38e-2, "C_H": 2.95e-4, "O_H": 5.37e-4, "N_H": 7.08e-5, "S_H": 1.41e-5,
}

# Mirror the VULCAN dataset-generation path (src/data_generation/generation.py):
# refractory minor elements get scaled by a GALAH-weighted [Fe/H] proxy from
# the C/O/S volatiles, with a Galactic-disk [alpha/Fe] correction.
_FASTCHEM_METALLICITY_SCALED_ELEMENTS = {
    "P", "Si", "Ti", "V", "Cl", "K", "Na", "Mg", "F", "Ca", "Fe",
}
_FEMH_VOLATILE_WEIGHTS = {"C_H": 0.48, "O_H": 0.31, "S_H": 0.21}
_ALPHA_FE_SLOPE = 0.15


def _metallicity_scale_from_globals(conditioning_inputs):
    volatile_dex = sum(
        w * np.log10(conditioning_inputs[name] / ML_SOLAR_ABUNDANCES[name])
        for name, w in _FEMH_VOLATILE_WEIGHTS.items()
    )
    return 10.0 ** float(volatile_dex / (1.0 - _ALPHA_FE_SLOPE))


def build_fastchem_reference_abundances(chem, conditioning_inputs):
    """Match the VULCAN FastChem runtime abundances used during training.

    The five conditioning elements (He, C, O, N, S) come from the caller.
    Refractory minors (P, Si, Ti, V, Cl, K, Na, Mg, F, Ca, Fe) are scaled
    by a GALAH-weighted [Fe/H] proxy so dataset-gen and this comparison
    use identical abundance profiles.
    """
    reference = np.asarray(chem.element_vector_reference, dtype=np.float64)
    h_index = chem.elements.index("H")
    fastchem_abundance = {
        element: float(reference[index] / reference[h_index])
        for index, element in enumerate(chem.elements)
        if element != "e-"
    }
    met_scale = _metallicity_scale_from_globals(conditioning_inputs)
    for raw_line in FASTCHEM_SOLAR_PATH.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        species_name, log_epsilon = line.split()[:2]
        if species_name in {"H", "e-"}:
            continue
        solar_value = 10 ** (float(log_epsilon) - 12.0)
        if species_name in _FASTCHEM_METALLICITY_SCALED_ELEMENTS:
            solar_value *= met_scale
        fastchem_abundance[species_name] = solar_value

    fastchem_abundance.update({
        "He": conditioning_inputs["He_H"],
        "C": conditioning_inputs["C_H"],
        "O": conditioning_inputs["O_H"],
        "N": conditioning_inputs["N_H"],
        "S": conditioning_inputs["S_H"],
    })
    return fastchem_abundance


In [ ]:
import jax.numpy as jnp

def renormalize_vmr_profile(vmr_profile):
    # The exported model predicts 17 species that should sum to ~1.0.
    return vmr_profile / jnp.sum(vmr_profile, axis=1, keepdims=True)

# Use ML model training abundances so inputs stay on the training manifold.
global_inputs = dict(ML_SOLAR_ABUNDANCES)

vmr = renormalize_vmr_profile(vmr_fn(Tarr, art.pressure, global_inputs))
idx = int(jnp.argmin(jnp.abs(art.pressure - 0.1)))
print(f"Eager inference: shape={vmr.shape}  H2O@0.1bar={float(vmr[idx, species_labels.index('H2O')]):.3e}")


In [ ]:
import jax

def vmr_fn_normalized(temperatures_k, pressures_bar, global_inputs):
    return renormalize_vmr_profile(vmr_fn(temperatures_k, pressures_bar, global_inputs))

vmr_jit = jax.jit(vmr_fn_normalized)
vmr_compiled = vmr_jit(Tarr, art.pressure, global_inputs)


ExoGibbs for comparison

In [ ]:
from exogibbs.presets.fastchem import chemsetup
from exogibbs.api.equilibrium import EquilibriumOptions, equilibrium_profile

chem = chemsetup(silent=True)
idx_co_exogibbs = chem.species.index("C1O1")
print("idx for CO=",idx_co_exogibbs, "JANAF name", chem.species[idx_co_exogibbs])
idx_h2_exogibbs = chem.species.index("H2")
print("idx for H2=",idx_h2_exogibbs, "JANAF name", chem.species[idx_h2_exogibbs])
print("element:", chem.elements)

# Match the VULCAN FastChem runtime abundances instead of ExoJAX's AAG21
# fallback so ExoGibbs is compared against the same chemistry reference.
fastchem_reference = build_fastchem_reference_abundances(chem, global_inputs)
element_vector = jnp.append(jnp.array([fastchem_reference[el] for el in chem.elements[:-1]]), 0.0)
print("element_vector:", element_vector)

Pref = 1.0
opts = EquilibriumOptions(epsilon_crit=1e-11, max_iter=1000, method="vmap_cold")

# equilibrium_profile accepts the ExoJAX top-to-bottom ordering directly.
res = equilibrium_profile(
    chem,
    Tarr,
    art.pressure,
    element_vector,
    Pref=Pref,
    options=opts,
)
nk_result = np.asarray(res.x)
vmr_co = nk_result[:, idx_co_exogibbs]
vmr_h2 = nk_result[:, idx_h2_exogibbs]


In [ ]:
idx_CO_vulcan = species_labels.index("CO") 
idx_H2_vulcan= species_labels.index("H2")
fig = plt.figure(figsize=(15, 5))
ax1 = fig.add_subplot(131)
plt.plot(Tarr, art.pressure)
ax1.invert_yaxis()
#ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Temperature (K)")
ax1.set_ylabel("Pressure (bar)")

ax2 = fig.add_subplot(132)
plt.plot(vmr_compiled[:, idx_H2_vulcan], art.pressure, label="Vulcan Emulator", alpha=0.75)
plt.plot(vmr_h2, art.pressure, label="ExoGibbs", alpha=0.75, ls="--")
ax2.invert_yaxis()
ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlabel("VMR (H2)")
ax2.set_xlim(1e-10, 2)
ax2.legend()

ax = fig.add_subplot(133)
plt.plot(vmr_compiled[:, idx_CO_vulcan], art.pressure, label="Vulcan Emulator", alpha=0.75)
plt.plot(vmr_co, art.pressure, label="ExoGibbs", alpha=0.75, ls="--")
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("VMR (CO)")
ax.legend()
plt.savefig("vmr_comparison.png", dpi=300)
plt.show()

The mass mixing ratio of CO (MMR) should be computed based on the thermochemical equilibirum.

In [ ]:
from exojax.atm.atmconvert import vmr_to_mmr
from exojax.database.molinfo.mass import isotope_molmass

idx_CO = species_labels.index("CO")
idx_H2 = species_labels.index("H2")

vmr_co = vmr_compiled[:, idx_CO]
vmr_h2 = vmr_compiled[:, idx_H2]

mean_molecular_weight = 2.33  ## assume constant (not accurate)
molmass = isotope_molmass("12C-16O")
mmr_profile = vmr_to_mmr(vmr_co, molmass, mean_molecular_weight)
mmr_profile_h2 = vmr_to_mmr(vmr_h2, isotope_molmass("1H2"), mean_molecular_weight)

import matplotlib.pyplot as plt

fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot(mmr_profile, art.pressure, label="CO (ML)")
ax.plot(mmr_profile_h2, art.pressure, ls="--", label="H2 (ML)")

ax.invert_yaxis()
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("mmr")
ax.set_ylabel("Pressure (bar)")
plt.show()


Surface gravity is also important quantity of the atmospheric model, which is a function of planetary radius and mass. Here we assume 1 RJ and 10 MJ.

In [ ]:
from exojax.utils.astrofunc import gravity_jupiter

gravity = gravity_jupiter(1.0, 10.0)

In addition to the CO cross section, we would consider [collisional induced absorption](https://en.wikipedia.org/wiki/Collision-induced_absorption_and_emission) (CIA) as a continuum opacity. `cdb` class can be used.

In [ ]:
from exojax.database.contdb  import CdbCIA
from exojax.opacity import OpaCIA

cdb = CdbCIA(".database/H2-H2_2011.cia", nurange=nu_grid)
opacia = OpaCIA(cdb, nu_grid=nu_grid)

Before running the radiative transfer, we need cross sections for layers, called `xsmatrix` for CO and `logacia_matrix` for CIA (strictly speaking, the latter is not cross section but coefficient because CIA intensity is proportional density square). See [here](CIA_opacity.html) for the details.

In [ ]:
xsmatrix = opa.xsmatrix(Tarr, art.pressure)
logacia_matrix = opacia.logacia_matrix(Tarr)

Convert them to opacity

In [ ]:
dtau_CO = art.opacity_profile_xs(xsmatrix, mmr_profile, mdb.molmass, gravity)
#vmrH2 = 0.855  # VMR of H2
dtaucia = art.opacity_profile_cia(logacia_matrix, Tarr, vmr_h2, vmr_h2, mean_molecular_weight, gravity)

Add two opacities.

In [ ]:
dtau = dtau_CO + dtaucia

Then, run the radiative transfer.
As you can see, the emission spectrum has been generated. This spectrum shows a region near 4360 cm-1, or around 22940 AA, where CO features become increasingly dense. This region is referred to as the band head. If you're interested in why the band head occurs, please refer to [Quatum states of Carbon Monoxide and Fortrat Diagram](Fortrat.html).


In [ ]:
F = art.run(dtau, Tarr)

fig = plt.figure(figsize=(15, 4))
plt.plot(nu_grid, F)
plt.xlabel("wavenumber (cm-1)")
plt.ylabel("flux (erg/s/cm2/cm-1)")
plt.show()

You can check the contribution function too! 
You should check if the dominant contribution is within the layer. 
If not, you need to change `pressure_top` and `pressure_btm` in `ArtEmisPure`    

In [ ]:
from exojax.plot.atmplot import plotcf

In [ ]:
cf = plotcf(nu_grid, dtau, Tarr, art.pressure, art.dParr)

## 4. Spectral Operators: rotational broadening, instrumental profile, Doppler velocity shift and so on, any operation on spectra.

The above spectrum is called "raw spectrum" in ExoJAX. The effects applied to the raw spectrum is handled in ExoJAX by the spectral operator (`sop`).
First, we apply the spin rotational broadening of a planet.

In [ ]:
from exojax.postproc.specop import SopRotation

sop_rot = SopRotation(nu_grid, vsini_max=100.0)

vsini = 10.0
u1 = 0.0
u2 = 0.0
Frot = sop_rot.rigid_rotation(F, vsini, u1, u2)

In [ ]:
fig = plt.figure(figsize=(15, 4))
plt.plot(nu_grid, F, label="raw spectrum")
plt.plot(nu_grid, Frot, label="rotated")
plt.xlabel("wavenumber (cm-1)")
plt.ylabel("flux (erg/s/cm2/cm-1)")
plt.legend()
plt.show()

Then, the instrumental profile with relative radial velocity shift is applied. Also, we need to match the computed spectrum to the data grid. This process is called `sampling` (but just interpolation though). Below, let’s perform a simulation that includes noise for use in later analysis.

In [ ]:
from exojax.postproc.specop import SopInstProfile
from exojax.utils.instfunc import resolution_to_gaussian_std

sop_inst = SopInstProfile(nu_grid, vrmax=1000.0)

RV = 40.0  # km/s
resolution_inst =70000.0
beta_inst = resolution_to_gaussian_std(resolution_inst)
Finst = sop_inst.ipgauss(Frot, beta_inst)
nu_obs = nu_grid[::5][:-50]


from numpy.random import normal
noise = 500.0
Fobs = sop_inst.sampling(Finst, RV, nu_obs) + normal(0.0, noise, len(nu_obs))

In [ ]:
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(211)
plt.plot(nu_grid, Frot, label="rotated")
plt.plot(nu_grid, Finst, label="rotated+IP")
plt.ylabel("flux (erg/s/cm2/cm-1)")
plt.legend()
ax = fig.add_subplot(212)
plt.errorbar(nu_obs, Fobs, noise, fmt=".", label="rotated + RV + IP (sampling)", color="gray",alpha=0.5)
plt.xlabel("wavenumber (cm-1)")
plt.legend()
plt.show()

## 5. Retrieval of an Emission Spectrum

Next, let’s perform a “retrieval” on the simulated spectrum created above. Retrieval involves estimating the parameters of an atmospheric model in the form of a posterior distribution based on the spectrum. To do this, we first need a model. Here, we have compiled the forward modeling steps so far and defined the model as follows. The spectral model samples the usual thermal/kinematic parameters plus the five elemental abundances (He/H, C/H, O/H, N/H, S/H) directly in log space — the same contract the FastChem emulator bundle was trained against.

In [ ]:
idx_CO = species_labels.index("CO")
idx_H2 = species_labels.index("H2")

def fspec(T0, alpha, g, RV, vsini, global_inputs_in):
    Tarr = art.powerlaw_temperature(T0, alpha)
    xsmatrix = opa.xsmatrix(Tarr, art.pressure)

    vmr = vmr_fn_normalized(Tarr, art.pressure, global_inputs_in)
    vmr_co = vmr[:, idx_CO]
    vmr_h2 = vmr[:, idx_H2]

    mmr_arr = vmr_to_mmr(vmr_co, molmass, mean_molecular_weight)
    dtau = art.opacity_profile_xs(xsmatrix, mmr_arr, molmass, g)
    logacia_matrix = opacia.logacia_matrix(Tarr)
    dtaucH2H2 = art.opacity_profile_cia(logacia_matrix, Tarr, vmr_h2, vmr_h2,
                                        mean_molecular_weight, g)
    dtau = dtau + dtaucH2H2
    F = art.run(dtau, Tarr)
    Frot = sop_rot.rigid_rotation(F, vsini, u1, u2)
    Finst = sop_inst.ipgauss(Frot, beta_inst)
    return sop_inst.sampling(Finst, RV, nu_obs)


Let’s verify that spectra are being generated from `fspec` with various parameter sets.

In [ ]:
fig = plt.figure(figsize=(12, 3))
plt.plot(nu_obs, fspec(1200.0, 0.09, gravity_jupiter(1.0, 1.0),  40.0, 10.0, global_inputs), label="T0=1200")
plt.plot(nu_obs, fspec(1100.0, 0.12, gravity_jupiter(1.0, 10.0), 20.0,  5.0, global_inputs), label="T0=1100")


NumPyro is a probabilistic programming language (PPL), which requires the definition of a probabilistic model. In the probabilistic model `model_prob` defined below, the prior distributions of each parameter are specified. The previously defined spectral model is used within this probabilistic model as a function that provides the mean $\mu$. The spectrum is assumed to be generated according to a Gaussian distribution with this mean and a standard deviation $\sigma$. i.e. $f(\nu_i) \sim \mathcal{N}(\mu(\nu_i; {\bf p}), \sigma^2 I)$, where ${\bf p}$ is the spectral model parameter set, which are the arguments of `fspec`. 



In [ ]:
from numpyro.infer import MCMC, NUTS
import numpyro.distributions as dist
import numpyro
from jax import random

In [ ]:
# (ExoGibbs element indices no longer needed; ML model used in fspec)


In [ ]:
def model_prob(spectrum):
    logg   = numpyro.sample("logg",   dist.Uniform(4.0, 5.0))
    RV     = numpyro.sample("RV",     dist.Uniform(35.0, 45.0))
    T0     = numpyro.sample("T0",     dist.Uniform(1000.0, 1500.0))
    alpha  = numpyro.sample("alpha",  dist.Uniform(0.05, 0.2))
    vsini  = numpyro.sample("vsini",  dist.Uniform(5.0, 15.0))

    # Elemental abundances sampled directly as log10(X/H). Priors are ±1 dex
    # around the solar-like training reference for each element.
    ref = ML_SOLAR_ABUNDANCES
    log_He_H = numpyro.sample("log_He_H",
                              dist.Uniform(jnp.log10(ref["He_H"]) - 0.3,
                                           jnp.log10(ref["He_H"]) + 0.3))
    log_C_H  = numpyro.sample("log_C_H",
                              dist.Uniform(jnp.log10(ref["C_H"]) - 1.0,
                                           jnp.log10(ref["C_H"]) + 1.0))
    log_O_H  = numpyro.sample("log_O_H",
                              dist.Uniform(jnp.log10(ref["O_H"]) - 1.0,
                                           jnp.log10(ref["O_H"]) + 1.0))
    log_N_H  = numpyro.sample("log_N_H",
                              dist.Uniform(jnp.log10(ref["N_H"]) - 1.0,
                                           jnp.log10(ref["N_H"]) + 1.0))
    log_S_H  = numpyro.sample("log_S_H",
                              dist.Uniform(jnp.log10(ref["S_H"]) - 1.0,
                                           jnp.log10(ref["S_H"]) + 1.0))

    gi = {
        "He_H": 10.0 ** log_He_H,
        "C_H":  10.0 ** log_C_H,
        "O_H":  10.0 ** log_O_H,
        "N_H":  10.0 ** log_N_H,
        "S_H":  10.0 ** log_S_H,
    }

    mu = fspec(T0, alpha, 10 ** logg, RV, vsini, gi)

    sigmain = numpyro.sample("sigmain", dist.Exponential(1.0e-3))
    numpyro.sample("spectrum", dist.Normal(mu, sigmain), obs=spectrum)

Note that we did not account for the effects of limb darkening. However, in actual analyses, one possible approach might be to use an uninformative prior, such as the one proposed by Kipping.

```python
    from exojax.postproc.limb_darkening import ld_kipping
    q1 = numpyro.sample('q1', dist.Uniform(0.0,1.0))
    q2 = numpyro.sample('q2', dist.Uniform(0.0,1.0))
    u1,u2 = ld_kipping(q1,q2)
```

Now, let’s define NUTS and start sampling. 

In [ ]:
rng_key = random.PRNGKey(0)
rng_key, rng_key_ = random.split(rng_key)
num_warmup, num_samples = 500, 1000
#kernel = NUTS(model_prob, forward_mode_differentiation=True)
kernel = NUTS(model_prob, forward_mode_differentiation=False)

Since this process will take several hours, feel free to go for a long lunch break!

In [ ]:
mcmc = MCMC(kernel, num_warmup=num_warmup, num_samples=num_samples)
mcmc.run(rng_key_, spectrum=Fobs)
mcmc.print_summary()

After returning from your long lunch, if you're lucky and the sampling is complete, let’s write a predictive model for the spectrum.

In [ ]:
from numpyro.diagnostics import hpdi
from numpyro.infer import Predictive
import jax.numpy as jnp

In [ ]:
# SAMPLING
posterior_sample = mcmc.get_samples()
pred = Predictive(model_prob, posterior_sample, return_sites=['spectrum'])
predictions = pred(rng_key_, spectrum=None)
median_mu1 = jnp.median(predictions['spectrum'], axis=0)
hpdi_mu1 = hpdi(predictions['spectrum'], 0.9)

In [ ]:

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(15, 4.5))
ax.plot(nu_obs, median_mu1, color='C1')
ax.fill_between(nu_obs,
                hpdi_mu1[0],
                hpdi_mu1[1],
                alpha=0.3,
                interpolate=True,
                color='C1',
                label='90% area')
ax.errorbar(nu_obs, Fobs, noise, fmt=".", label="mock spectrum", color="black",alpha=0.5)
plt.xlabel('wavenumber (cm-1)', fontsize=16)
plt.legend(fontsize=14)
plt.tick_params(labelsize=14)
plt.show()

In [ ]:
#save the result
import arviz
idata = arviz.from_numpyro(mcmc,
    posterior_predictive=predictions, coords = {"wavenumber": nu_obs,},dims = {"spectrum": ["wavenumber"],})
arviz.to_netcdf(idata, "posterior_elements.nc")

You can see that the predictions are working very well! Let’s also display a corner plot. Here, we’ve used ArviZ for visualization.

In [ ]:
import arviz
pararr = ['T0', 'alpha', 'logg', 'log_He_H', 'log_C_H', 'log_O_H', 'log_N_H', 'log_S_H', 'vsini', 'RV']
arviz.plot_pair(arviz.from_numpyro(mcmc),
                var_names=pararr,
                kind='kde',
                divergences=False,
                marginals=True)
plt.show()

Expect strong degeneracies between gravity and the bulk-metal abundances (especially log_O_H and log_C_H) — the gravity column density trades off against the total absorber column in the photosphere.